In [ ]:
# ============================================================
#  Project Name: brandalyze
#  File: tweet_analysis.ipynb
#  Author: dngi (https://twitter.com/_dngi)
#  Created: <2026-02-13>
#  
#  Copyright (c) 2026 Dom G. (https://twitter.com/_dngi)
#  All rights reserved.
#
#  This notebook and its contents are confidential and
#  proprietary. Unauthorized copying, distribution,
#  modification, or use is strictly prohibited.
# ============================================================

# Brandalyze Viral Tweet Analysis

In [ ]:
# imports
import json
import pandas as pd
import numpy as np

## Data Loading/Exploration

In [ ]:
# load dataset
with open('labeled-viral-tweets.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# convert to DataFrame for easier analysis
df = pd.DataFrame(data)

# basic info
print(f"Total tweets: {len(df)}")
print(f"Labeled tweets: {df['labels'].notna().sum()}")

# preview
df.head()

## Label Data Extraction

In [ ]:
# extract label fields into separate columns
df['format'] = df['labels'].apply(lambda x: x.get('format') if isinstance(x, dict) else None)
df['hookQuality'] = df['labels'].apply(lambda x: x.get('hookQuality') if isinstance(x, dict) else None)
df['closerType'] = df['labels'].apply(lambda x: x.get('closerType') if isinstance(x, dict) else None)

# check distribution
print("\nFormat distribution:")
print(df['format'].value_counts())
print("\nHook quality distribution:")
print(df['hookQuality'].value_counts())
print("\nCloser type distribution:")
print(df['closerType'].value_counts())

## Engagement Metric Extraction
- likes, retweets, views bookmarks
- calculate engagement rates
- add virality tiers

In [ ]:
df['likes'] = df['likeCount']
df['retweets'] = df['retweetCount']
df['replies'] = df['replyCount']
df['views'] = df['viewCount']
df['bookmarks'] = df['bookmarkCount']

df['total_engagement'] = df['likes'] + df['retweets'] + df['replies'] + df['bookmarks']
df['engagement_rate'] = (df['total_engagement'] / df['views']) * 100

print(df['viralityTier'].value_counts())

## Author Metric Extraction
- get follower counts, following, bio
- categorize accounts by size
- extract tweet type (e.g. tweet, quote)

In [ ]:
df['author_username'] = df['author'].apply(lambda x: x.get('userName'))
df['author_followers'] = df['author'].apply(lambda x: x.get('followers'))
df['author_bio'] = df['author'].apply(lambda x: x.get('description'))

# categorize by follower count
def categorize_account_size(followers):
    if followers < 5000:
        return 'micro'
    elif followers < 50000:
        return 'mid'
    else:
        return 'macro'

df['account_size'] = df['author_followers'].apply(categorize_account_size)

# extract tweet type
df['is_quote'] = df['isQuote']
df['is_reply'] = df['isReply']
df['tweet_type'] = df.apply(lambda x: 'quote' if x['is_quote'] else ('reply' if x['is_reply'] else 'tweet'), axis=1)

## Text Analysis Prep
- extract tweet text
- calculate text_length
- check for media presence
- count hashtags/mentions/URLs

In [ ]:
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

df['has_media'] = df['extendedEntities'].apply(lambda x: bool(x.get('media', [])) if isinstance(x, dict) else False)
df['media_type'] = df['extendedEntities'].apply(
    lambda x: x.get('media', [{}])[0].get('type') if isinstance(x, dict) and x.get('media') else None
)

# Entity counts
df['hashtag_count'] = df['entities'].apply(lambda x: len(x.get('hashtags', [])) if isinstance(x, dict) else 0)
df['mention_count'] = df['entities'].apply(lambda x: len(x.get('user_mentions', [])) if isinstance(x, dict) else 0)
df['url_count'] = df['entities'].apply(lambda x: len(x.get('urls', [])) if isinstance(x, dict) else 0)

## Correlation Analysis
- format vs engagement
- hook quality vs virality
- closer type vs engagement
- account size impact

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# format performance
format_stats = df.groupby('format').agg({
    'likes': 'mean',
    'total_engagement': 'mean',
    'engagement_rate': 'mean'
}).round(2)

print("Average engagement by format:")
print(format_stats)

# hook quality impact
hook_stats = df.groupby('hookQuality').agg({
    'likes': 'mean',
    'views': 'mean',
    'engagement_rate': 'mean'
}).round(2)

print("\nEngagement by hook quality:")
print(hook_stats)

## Visualizations
- bar charts for format/hook